In [0]:
%run /Shared/insclm_capstone/NB_00_config_loader.py

[SecretScope(name=' kv-insclm-cap-11'), SecretScope(name='kv-insclm')]

[SecretMetadata(key='adls-abfss-base'),
 SecretMetadata(key='adls-account-key'),
 SecretMetadata(key='adls-account-name'),
 SecretMetadata(key='adls-audit-path'),
 SecretMetadata(key='adls-base-url'),
 SecretMetadata(key='adls-bronze-path'),
 SecretMetadata(key='adls-container-name'),
 SecretMetadata(key='adls-gold-path'),
 SecretMetadata(key='adls-raw-path'),
 SecretMetadata(key='adls-rejected-path'),
 SecretMetadata(key='adls-silver-path'),
 SecretMetadata(key='database-workspace-url'),
 SecretMetadata(key='databricks-cluster-id'),
 SecretMetadata(key='databricks-pat'),
 SecretMetadata(key='file-claim-status-updates'),
 SecretMetadata(key='file-claims'),
 SecretMetadata(key='file-customer-master'),
 SecretMetadata(key='file-policy-master'),
 SecretMetadata(key='github-pat'),
 SecretMetadata(key='github-repo-url'),
 SecretMetadata(key='sql-admin-name'),
 SecretMetadata(key='sql-admin-password'),
 SecretMetadata(key='sql-connection-string'),
 SecretMetadata(key='sql-database-name'),
 S

✅ Config loaded from Key Vault successfully.
   ADLS Account  : [REDACTED]
   Container     : [REDACTED]
   ABFSS Base    : [REDACTED]
   RAW path      : [REDACTED][REDACTED]
   BRONZE path   : [REDACTED][REDACTED]
   SILVER path   : [REDACTED][REDACTED]
   GOLD path     : [REDACTED][REDACTED]
   REJECTED path : [REDACTED][REDACTED]
   AUDIT path    : [REDACTED][REDACTED]
   SQL Server    : [REDACTED]
   SQL Database  : [REDACTED]


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

print("=" * 55)
print("DELTA LAKE TIME TRAVEL DEMONSTRATION")
print("=" * 55)

DELTA LAKE TIME TRAVEL DEMONSTRATION


In [0]:
print("\n📋 Full history of silver_policy_dim:")

DeltaTable.forName(
    spark, "silver_insclm.silver_policy_dim") \
    .history() \
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    ).show(10, truncate=False)


📋 Full history of silver_policy_dim:
+-------+-------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation                        |operationMetrics                                                                                                                           |
+-------+-------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------+
|0      |2026-05-21 14:32:47|CREATE OR REPLACE TABLE AS SELECT|{numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1500, numOutputBytes -> 61463}|
+-------+-------------------+---------------------------------+-----------------------------------------------------------------------------------

In [0]:
print("\n📋 Time Travel — by VERSION number")
print("   Reading version 0 = first load\n")

v0_df = spark.read \
    .format("delta") \
    .option("versionAsOf", 0) \
    .table("silver_insclm.silver_policy_dim")

v0_count      = v0_df.count()
current_count = spark.table(
    "silver_insclm.silver_policy_dim").count()

print(f"   Version 0 row count : {v0_count:,}")
print(f"   Current row count   : {current_count:,}")
print(f"   Difference          : {current_count - v0_count:,}")

print("\nSample rows from Version 0:")
v0_df.select(
    "policy_id", "policy_type",
    "policy_status", "coverage_amount", "is_current"
).show(5, truncate=False)


📋 Time Travel — by VERSION number
   Reading version 0 = first load

   Version 0 row count : 1,500
   Current row count   : 1,500
   Difference          : 0

Sample rows from Version 0:
+---------+-----------+-------------+---------------+----------+
|policy_id|policy_type|policy_status|coverage_amount|is_current|
+---------+-----------+-------------+---------------+----------+
|POL000001|Travel     |Cancelled    |8498893.00     |true      |
|POL000002|Home       |Active       |5067770.50     |true      |
|POL000003|Travel     |Lapsed       |3952137.25     |true      |
|POL000004|Home       |Active       |2411665.75     |true      |
|POL000005|Home       |Active       |7077523.00     |true      |
+---------+-----------+-------------+---------------+----------+
only showing top 5 rows


In [0]:
print("\n📋 Time Travel — by TIMESTAMP\n")

# Get the earliest timestamp from Delta history
history_df = DeltaTable.forName(
    spark, "silver_insclm.silver_policy_dim") \
    .history()

earliest_ts = history_df.agg(
    F.min("timestamp")).first()[0]

print(f"   Earliest version timestamp: {earliest_ts}")

try:
    ts_df = spark.read \
        .format("delta") \
        .option("timestampAsOf", str(earliest_ts)) \
        .table("silver_insclm.silver_policy_dim")

    print(f"   Rows at earliest version: {ts_df.count():,}")

    ts_df.select(
        "policy_id", "policy_status",
        "coverage_amount", "is_current"
    ).show(5, truncate=False)

except Exception as e:
    print(f"   Note: {e}")


📋 Time Travel — by TIMESTAMP

   Earliest version timestamp: 2026-05-21 14:32:47
   Rows at earliest version: 1,500
+---------+-------------+---------------+----------+
|policy_id|policy_status|coverage_amount|is_current|
+---------+-------------+---------------+----------+
|POL000001|Cancelled    |8498893.00     |true      |
|POL000002|Active       |5067770.50     |true      |
|POL000003|Lapsed       |3952137.25     |true      |
|POL000004|Active       |2411665.75     |true      |
|POL000005|Active       |7077523.00     |true      |
+---------+-------------+---------------+----------+
only showing top 5 rows


In [0]:
print("\n📋 Policy status at claim date vs current status\n")

current_policy = (spark
    .table("silver_insclm.silver_policy_dim")
    .filter(F.col("is_current") == True)
    .select(
        "policy_id",
        F.col("policy_status").alias("current_status"),
        F.col("coverage_amount").alias("current_coverage")
    ))

claims_snapshot = (spark
    .table("silver_insclm.silver_claims_fact")
    .select(
        "claim_id",
        "policy_id",
        "claim_date",
        "policy_status_at_claim",
        "claim_amount"
    ).distinct())

comparison = (claims_snapshot
    .join(current_policy, "policy_id", "left")
    .withColumn("status_changed",
        F.col("policy_status_at_claim") !=
        F.col("current_status")))

total         = comparison.count()
changed_count = comparison.filter(
    F.col("status_changed") == True).count()
unchanged     = total - changed_count

print(f"   Total claims compared    : {total:,}")
print(f"   Status same as today     : {unchanged:,}")
print(f"   Status changed since claim: {changed_count:,}")

if changed_count > 0:
    print("\nSample — policies that changed after claim:")
    comparison \
        .filter(F.col("status_changed") == True) \
        .select(
            "claim_id", "policy_id", "claim_date",
            "policy_status_at_claim", "current_status"
        ).show(10, truncate=False)
else:
    print("\nNo policy status changes detected")


📋 Policy status at claim date vs current status

   Total claims compared    : 2,080
   Status same as today     : 2,080
   Status changed since claim: 0

No policy status changes detected


In [0]:
print("\n📋 Delta history of silver_claims_fact:")

DeltaTable.forName(
    spark, "silver_insclm.silver_claims_fact") \
    .history() \
    .select(
        "version", "timestamp",
        "operation", "operationMetrics"
    ).show(5, truncate=False)

print("\n📋 Delta history of gold_claim_summary:")

DeltaTable.forName(
    spark, "gold_insclm.gold_claim_summary") \
    .history() \
    .select(
        "version", "timestamp",
        "operation", "operationMetrics"
    ).show(5, truncate=False)


📋 Delta history of silver_claims_fact:
+-------+-------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation                        |operationMetrics                                                                                                                           |
+-------+-------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------+
|0      |2026-05-21 15:03:39|CREATE OR REPLACE TABLE AS SELECT|{numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 2080, numOutputBytes -> 64233}|
+-------+-------------------+---------------------------------+---------------------------------------------------------------------------------

In [0]:
print("\n" + "=" * 55)
print("TIME TRAVEL SUMMARY")
print("=" * 55)
print("✅ DESCRIBE HISTORY — silver_policy_dim")
print("✅ Time Travel by version — demonstrated")
print("✅ Time Travel by timestamp — demonstrated")
print("✅ Policy status at claim date — compared")
print("✅ DESCRIBE HISTORY — silver_claims_fact")
print("✅ DESCRIBE HISTORY — gold_claim_summary")
print("=" * 55)
print("✅ NB_07 complete.")
print("   Next: Run NB_08_describe_history_optimize")


TIME TRAVEL SUMMARY
✅ DESCRIBE HISTORY — silver_policy_dim
✅ Time Travel by version — demonstrated
✅ Time Travel by timestamp — demonstrated
✅ Policy status at claim date — compared
✅ DESCRIBE HISTORY — silver_claims_fact
✅ DESCRIBE HISTORY — gold_claim_summary
✅ NB_07 complete.
   Next: Run NB_08_describe_history_optimize
